# Black-Litterman, applied to the World Cup

This notebook walks through the idea and the maths. The one-line thesis: **Black-Litterman is not really a finance technique, it is a recipe for blending a defensible baseline with subjective views** — and football, where human judgement still beats pure models, is a perfect place to show it.

| Black-Litterman in finance | Here |
|---|---|
| Equilibrium expected returns | Statistical team-strength ratings |
| Covariance of returns | Uncertainty in each rating |
| Investor views (P, Q, Ω) | "I rate this team higher/lower", with confidence |
| Portfolio weights from the optimiser | Win probabilities from the Monte-Carlo simulator |


## 1. Setup

In [1]:
import numpy as np, pandas as pd
import config
from ratings import build_default_model, load_matches
from simulate import Simulator
from blacklitterman import BlackLitterman
from house_views import HOUSE_VIEWS, apply_house_views
from injuries import INJURY_VIEWS
from underlying_xg import UNDERLYING_XG_VIEWS
from market import market_probs
TEAMS=[t for g in config.GROUPS.values() for t in g]
print(len(TEAMS),'teams')

48 teams


## 2. The statistical baseline (the equilibrium)

A time-decayed Dixon-Coles goals model (recent matches weighted far more, friendlies discounted) blended with Elo, plus a squad-quality signal (market value x EA FC ratings). This is the *prior* — what we believe before imposing any judgement.

In [2]:
model = build_default_model()
S = model.strength()
top = sorted(TEAMS, key=lambda t: S[t], reverse=True)[:10]
pd.DataFrame({'team':top,'strength':[round(S[t],2) for t in top],
             'Elo':[round(model.elo[t]) for t in top]})

,team,strength,Elo
0,Spain,3.13,2018
1,Argentina,3.12,1965
2,England,3.08,1926
3,France,3.05,1946
4,Brazil,2.80,1861
5,Portugal,2.68,1873
6,Germany,2.48,1840
7,Netherlands,2.47,1851
8,Morocco,2.46,1903
9,Ecuador,2.38,1802


## 3. From ratings to title odds

The simulator plays the real 2026 bracket thousands of times. Like a portfolio optimiser turning expected returns into weights, it turns ratings into probabilities that sum to 100%.

In [3]:
sim = Simulator(model)
base = sim.monte_carlo(n_sims=5000, seed=2026).set_index('team')['win']
base.sort_values(ascending=False).head(8).mul(100).round(1)

team
Spain        20.5
Argentina    14.9
France       13.2
England      11.8
Brazil        7.2
Portugal      6.6
Germany       5.1
Morocco       3.2
Name: win, dtype: float64

## 4. The Black-Litterman master formula

$$\mu_{BL} = \left[(\tau\Sigma)^{-1} + P^{\top}\Omega^{-1}P\right]^{-1}\left[(\tau\Sigma)^{-1}\pi + P^{\top}\Omega^{-1}Q\right]$$

- $\pi$ = baseline strength ratings (the equilibrium)
- $\Sigma$ = uncertainty in those ratings, $\tau$ a scaling scalar
- $P, Q$ = the views (which teams, what target), $\Omega$ = how confident
- $\mu_{BL}$ = posterior ratings, which we feed back into the simulator

We apply it to **ratings** (unbounded), not probabilities, so the update is well-posed; the simulator restores the sum-to-one constraint.

## 5. Imposing views

Our editable, sourced views: dark horses (Morocco, Ecuador), an ageing-Argentina discount, France-to-deliver, injuries, underlying xG. Watch the posterior move.

In [4]:
bl = BlackLitterman(TEAMS, S, model.strength_uncertainty(), tau=0.10)
apply_house_views(bl, HOUSE_VIEWS + INJURY_VIEWS + UNDERLYING_XG_VIEWS)
shift, post, post_std = bl.solve()
full = sim.monte_carlo(n_sims=5000, shift=shift, seed=2026).set_index('team')['win']
cmp = pd.DataFrame({'baseline':base.mul(100), 'with views':full.mul(100)}).round(1)
cmp.sort_values('with views', ascending=False).head(8)

,baseline,with views
team,,
Spain,20.5,21.6
France,13.2,16.7
England,11.8,11.7
Argentina,14.9,11.1
Portugal,6.6,7.1
Brazil,7.2,5.6
Morocco,3.2,4.5
Germany,5.1,4.1


## 6. Validation — does the baseline actually work?

Backtest on the 2018 and 2022 World Cups: train only on prior matches, predict the tournament, score against an Elo-only baseline. Lower is better.

In [5]:
from backtest_tournaments import backtest_year, TOURNAMENTS
rows=[backtest_year(y,c) for y,c in TOURNAMENTS.items()]
pd.DataFrame(rows)[['year','matches','model_ll','base_ll','model_rps','base_rps']].round(4)

,year,matches,model_ll,base_ll,model_rps,base_rps
0,2018,64,0.9608,1.0257,0.2015,0.2253
1,2022,64,1.0524,1.0642,0.2171,0.2269


## 7. Sensitivity — owning the judgement calls

The baseline is validated; the squad weight, softening and views are judgement. Here is exactly how much the squad weight moves the headline, so nothing is a hidden lever.

In [6]:
out=[]
for w in [0.0,0.25,0.5,0.75]:
    m=build_default_model(squad_weight=w)
    p=Simulator(m).monte_carlo(n_sims=4000,seed=42).set_index('team')['win']
    out.append({'squad_weight':w, **{t:round(p[t]*100,1) for t in ['Spain','France','Argentina','England']}})
pd.DataFrame(out)

,squad_weight,Spain,France,Argentina,England
0,0.00,12.6,8.0,14.4,7.9
1,0.25,16.2,11.5,15.1,10.9
2,0.50,19.3,12.5,16.6,13.0
3,0.75,22.7,16.4,14.7,15.0


## 8. Model vs Market

The market is the sharpest external benchmark. We compare rather than copy; our value is the transparent, adjustable decomposition.

In [7]:
mkt=market_probs(TEAMS)
order=full.sort_values(ascending=False).head(8).index
pd.DataFrame({'model+views':[round(full[t]*100,1) for t in order],
             'market':[round(mkt[t]*100,1) for t in order]}, index=order)

,model+views,market
team,,
Spain,21.6,14.2
France,16.7,17.0
England,11.7,10.6
Argentina,11.1,8.5
Portugal,7.1,9.4
Brazil,5.6,7.7
Morocco,4.5,2.1
Germany,4.1,5.7


## Conclusion

The **baseline is evidence** (it beats Elo out of sample); the **overlay is opinion** (sourced, editable, and sensitivity-tested). That separation is the whole point of Black-Litterman, and it is the honest way to build a model with subjective inputs. The same discipline — knowing which numbers are estimated and which are imposed — is exactly what a risk or portfolio model demands.